In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("creditcard.csv")
normal = df[df["Class"] == 0].reset_index(drop=True)
fraud = df[df["Class"] == 1].reset_index(drop=True)
features = normal.drop(["Class"], axis=1)
fraud_features = fraud.drop(["Class"], axis=1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(features)
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled)

In [ ]:
bins = 16
x_bins = np.linspace(0, 1, bins + 1)
y_bins = np.linspace(0, 1, bins + 1)
H, _, _ = np.histogram2d(pca_features[:, 0], pca_features[:, 1], bins=[x_bins, y_bins])
target_dist = H.flatten() / np.sum(H)


# cell 4
def block_entangled_qcbm(params, n_qubits=8, layers=5):
qc = QuantumCircuit(n_qubits)
idx = 0
for _ in range(layers):
for q in range(n_qubits):
qc.ry(params[idx % len(params)], q)
idx += 1
for q in range(0, n_qubits, 2):
qc.cx(q, q + 1)
return qc

In [ ]:
def get_distribution_from_params(params, n_qubits=8, layers=5):
qc = block_entangled_qcbm(params, n_qubits=n_qubits, layers=layers)
sv = Statevector.from_instruction(qc)
probs = np.abs(sv.data) ** 2
return probs

In [ ]:
def cost(params, target):
dist = get_distribution_from_params(params)
return np.sum((dist - target) ** 2)

In [ ]:
np.random.seed(42)
param_count = 8 * 5
params = np.random.uniform(0, np.pi, param_count)
best_params = params.copy()
best_cost = cost(best_params, target_dist)

In [ ]:
for epoch in range(200):
proposal = best_params + np.random.normal(0, 0.2, size=best_params.shape)
c = cost(proposal, target_dist)
if c < best_cost:
best_cost = c
best_params = proposal.copy()


learned_dist = get_distribution_from_params(best_params)

In [ ]:
pca_fraud = pca.transform(scaler.transform(fraud_features))
num_eval_normals = min(len(pca_fraud), len(pca_features))
pca_normals_for_eval = pca_features[:num_eval_normals]


def get_likelihood(x, y):
xi = np.clip(np.digitize(x, x_bins) - 1, 0, bins - 1)
yi = np.clip(np.digitize(y, y_bins) - 1, 0, bins - 1)
idx = xi * bins + yi
return learned_dist[idx]


fraud_scores = [get_likelihood(x, y) for x, y in pca_fraud]
normal_scores = [get_likelihood(x, y) for x, y in pca_normals_for_eval]


threshold = np.percentile(normal_scores, 10)


y_true = np.array([1] * len(fraud_scores) + [0] * len(normal_scores))
y_pred = np.array([1 if s < threshold else 0 for s in fraud_scores + normal_scores])


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc = roc_auc_score(y_true, y_pred)


print(precision, recall, f1, roc)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
plt.bar(range(len(learned_dist)), learned_dist)
plt.title('Learned distribution (flattened)')
plt.show()


plt.figure(figsize=(6,5))
plt.hist(normal_scores, bins=50, alpha=0.6, label='normal')
plt.hist(fraud_scores, bins=50, alpha=0.6, label='fraud')
plt.axvline(threshold, color='k', linestyle='--')
plt.legend()
plt.title('Likelihood scores')
plt.show()